# Bayesian Inference from Scratch

In this notebook, we'll explore Bayesian inference techniques and implement them from scratch. Bayesian statistics provides a framework for updating our beliefs as new evidence emerges, making it a powerful tool for machine learning.

## Learning Objectives
- Understand Bayes' theorem and its implications
- Implement a Bayesian model from scratch
- Apply Bayesian inference to classification problems
- Visualize prior and posterior distributions

In [ ]:
import sys
import math
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Add the parent directory to the path so we can import our modules
sys.path.append('../..')
from phase0.statistics.bayesian_inference import BayesianModel

# Set the style for our plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Bayes' Theorem: The Foundation

Bayes' theorem is the cornerstone of Bayesian inference. It allows us to update our beliefs based on new evidence. Mathematically, it is expressed as:

$$P(H|E) = \frac{P(E|H) \times P(H)}{P(E)}$$

Where:
- $P(H|E)$ is the posterior probability: the probability of hypothesis $H$ given evidence $E$
- $P(E|H)$ is the likelihood: the probability of observing evidence $E$ given that hypothesis $H$ is true
- $P(H)$ is the prior probability: our initial belief about hypothesis $H$ before seeing the evidence
- $P(E)$ is the marginal likelihood: the total probability of observing evidence $E$ under all possible hypotheses

Let's illustrate this with a simple example:

In [ ]:
def bayes_theorem(prior, likelihood, evidence):
    """Calculate the posterior probability using Bayes' theorem."""
    return (likelihood * prior) / evidence

# Example: Medical test for a rare disease
# Given information
disease_prevalence = 0.01  # Prior: 1% of population has the disease
test_sensitivity = 0.95    # Likelihood: P(positive test | has disease) = 95%
test_specificity = 0.90    # P(negative test | no disease) = 90%

# Calculate P(test positive) - the evidence
p_test_positive = (test_sensitivity * disease_prevalence) + ((1 - test_specificity) * (1 - disease_prevalence))

# Calculate posterior: P(has disease | test positive)
p_disease_given_positive = bayes_theorem(disease_prevalence, test_sensitivity, p_test_positive)

print(f"Prior probability of having the disease: {disease_prevalence:.1%}")
print(f"Probability of testing positive if you have the disease: {test_sensitivity:.1%}")
print(f"Probability of testing positive if you don't have the disease: {1-test_specificity:.1%}")
print(f"Overall probability of testing positive: {p_test_positive:.1%}")
print(f"Posterior probability - If you test positive, probability you have the disease: {p_disease_given_positive:.1%}")

The result might be surprising! Even if the test is 95% accurate, if you test positive, there's still a relatively low chance that you actually have the disease. This is due to the low prior probability (disease prevalence) and demonstrates why understanding Bayesian reasoning is so important.

## 2. Using Our BayesianModel Class

Now, let's use our `BayesianModel` class to solve the same problem:

In [ ]:
# Create a Bayesian model for the medical test example
medical_test = BayesianModel()

# Set prior probabilities: P(disease), P(no disease)
medical_test.set_prior({
    'disease': 0.01,
    'no_disease': 0.99
})

# Set likelihoods: P(test result | disease status)
medical_test.set_likelihood({
    'disease': {
        'positive': 0.95,  # P(positive | disease)
        'negative': 0.05   # P(negative | disease)
    },
    'no_disease': {
        'positive': 0.10,  # P(positive | no disease)
        'negative': 0.90   # P(negative | no disease)
    }
})

# Update with a positive test result
posterior = medical_test.update('positive')

print("Posterior probabilities after a positive test:")
for hypothesis, probability in posterior.items():
    print(f"P({hypothesis} | positive) = {probability:.1%}")

Let's visualize how our beliefs update with sequential evidence:

In [ ]:
# Reset the model
medical_test = BayesianModel()
medical_test.set_prior({
    'disease': 0.01,
    'no_disease': 0.99
})
medical_test.set_likelihood({
    'disease': {
        'positive': 0.95,
        'negative': 0.05
    },
    'no_disease': {
        'positive': 0.10,
        'negative': 0.90
    }
})

# Track belief updates through multiple tests
disease_prob = [medical_test.prior['disease']]
test_results = ['positive', 'positive', 'negative', 'positive', 'positive']

for result in test_results:
    medical_test.update(result)
    disease_prob.append(medical_test.posterior['disease'])

# Plotting how our belief changes
plt.figure(figsize=(12, 6))
plt.plot(range(len(disease_prob)), disease_prob, 'bo-', linewidth=2, markersize=10)
plt.xticks(range(len(disease_prob)), ['Prior'] + test_results)
plt.title('Bayesian Belief Update: Probability of Having the Disease')
plt.xlabel('Evidence (Test Results)')
plt.ylabel('Probability')
plt.ylim(0, 1)
plt.grid(True)

# Add annotations
for i, prob in enumerate(disease_prob):
    plt.annotate(f'{prob:.4f}', (i, prob), textcoords="offset points", 
                 xytext=(0,10), ha='center')
    
plt.show()

Notice how our belief in the hypothesis 'having the disease' increases with positive test results and decreases with negative ones. The more evidence we gather, the more confident we become in our belief.

## 3. Bayesian Parameter Estimation

Now let's apply Bayesian reasoning to a slightly more complex problem: estimating the bias of a coin.

Imagine we have a coin that might be biased (probability of heads not equal to 0.5). We'll use Bayesian inference to update our belief about the bias parameter $\theta$ as we observe coin tosses.

In [ ]:
class BetaBinomialModel:
    """Beta-Binomial model for coin flip experiments."""
    
    def __init__(self, alpha=1, beta=1):
        """Initialize with Beta(alpha, beta) prior."""
        self.alpha = alpha
        self.beta = beta
    
    def update(self, heads, tails):
        """Update model with new observations."""
        self.alpha += heads
        self.beta += tails
    
    def posterior_mean(self):
        """Expected value of the posterior distribution."""
        return self.alpha / (self.alpha + self.beta)
    
    def posterior_mode(self):
        """Mode of the posterior distribution."""
        if self.alpha > 1 and self.beta > 1:
            return (self.alpha - 1) / (self.alpha + self.beta - 2)
        else:
            # Mode not defined for some parameter values
            return None
    
    def credible_interval(self, p=0.95):
        """Calculate credible interval for theta."""
        # Use scipy's beta distribution
        dist = stats.beta(self.alpha, self.beta)
        lower = dist.ppf((1-p)/2)
        upper = dist.ppf(1-(1-p)/2)
        return (lower, upper)
    
    def plot_posterior(self):
        """Plot the posterior distribution."""
        theta = np.linspace(0, 1, 1000)
        dist = stats.beta(self.alpha, self.beta)
        pdf = dist.pdf(theta)
        
        plt.figure(figsize=(12, 6))
        plt.plot(theta, pdf, 'b-', linewidth=2)
        plt.fill_between(theta, pdf, alpha=0.3)
        plt.axvline(self.posterior_mean(), color='r', linestyle='--', label='Mean')
        
        mode = self.posterior_mode()
        if mode is not None:
            plt.axvline(mode, color='g', linestyle='--', label='Mode')
        
        # Add credible interval
        ci = self.credible_interval()
        plt.axvline(ci[0], color='k', linestyle=':', label=f'95% CI')
        plt.axvline(ci[1], color='k', linestyle=':')
        plt.fill_between(theta, pdf, where=((theta>=ci[0]) & (theta<=ci[1])), 
                         color='gray', alpha=0.3)
        
        plt.title(f'Beta({self.alpha}, {self.beta}) Posterior Distribution')
        plt.xlabel('θ (Probability of Heads)')
        plt.ylabel('Density')
        plt.legend()
        plt.grid(True)
        plt.show()
        
        return theta, pdf

Let's try a coin flipping experiment. We'll start with a uniform prior (Beta(1,1)) and update as we observe coin flips.

In [ ]:
# Create a model with uniform prior
coin_model = BetaBinomialModel(alpha=1, beta=1)

# Plot the prior
coin_model.plot_posterior()

In [ ]:
# Let's simulate 10 flips from a biased coin with true p(heads) = 0.7
true_p = 0.7
np.random.seed(42)  # For reproducibility
flips = np.random.choice(['H', 'T'], size=10, p=[true_p, 1-true_p])
print("Flips:", ", ".join(flips))

# Update the model
heads = sum(flip == 'H' for flip in flips)
tails = sum(flip == 'T' for flip in flips)
coin_model.update(heads, tails)

# Plot the posterior after 10 flips
coin_model.plot_posterior()
print(f"After {heads+tails} flips ({heads} heads, {tails} tails):")
print(f"Posterior mean: {coin_model.posterior_mean():.4f}")
print(f"95% credible interval: {coin_model.credible_interval()}")

In [ ]:
# Let's observe 100 more flips
more_flips = np.random.choice(['H', 'T'], size=100, p=[true_p, 1-true_p])
heads = sum(flip == 'H' for flip in more_flips)
tails = sum(flip == 'T' for flip in more_flips)
coin_model.update(heads, tails)

# Plot the posterior after 110 flips
coin_model.plot_posterior()
print(f"After 110 flips (in total):")
print(f"Posterior mean: {coin_model.posterior_mean():.4f}")
print(f"95% credible interval: {coin_model.credible_interval()}")

Notice how our posterior distribution becomes more concentrated around the true value of θ = 0.7 as we gather more data. This is a key feature of Bayesian inference: we continuously update our beliefs with new evidence.

## Conclusion

In this notebook, we've explored the fundamentals of Bayesian inference. We've learned:

1. How to apply Bayes' theorem to update our beliefs with new evidence
2. How to implement Bayesian parameter estimation
3. How to visualize prior and posterior distributions

In the next notebook, we'll explore Bayesian classification and more advanced applications of Bayesian methods.